# Experiments: model v2, CEW external test, EAR comparison, model v3

Google Colab (T4 GPU). Data and models are read from Google Drive (`/content/drive/MyDrive/eye_project`).

| Cell | What it does |
|---|---|
| 1 | Setup: install MediaPipe, mount Drive, unzip our dataset and CEW |
| 2 | Model v2: training with blur / low-resolution augmentation, test on our test set and on CEW |
| 3 | Webcam capture helper |
| 4 | CEW: 100 × 100 vs original-resolution closed faces |
| 5 | EAR baseline vs CNN on CEW |
| 6 | Hybrid EAR + CNN with 5-fold cross-validation (needs `LogisticRegression`, `cross_val_predict` and `cv` from an earlier cell) |
| 7 | CNN trained on 70% of CEW, compared on the other 30% |
| 8 | **Final model v3**: our data + 70% of CEW, tested on both test sets |

Printed results are kept; images were removed. The v2 training run in cell 2 was stopped early in this copy; its full results are in the README.


In [ ]:
!pip install -q mediapipe

from google.colab import drive
drive.mount('/content/drive')

# our dataset
!unzip -q -o "/content/drive/MyDrive/eye_project/my_dataset/Mydataset.zip" -d /content/my_dataset
# CEW
!unzip -q -o "/content/drive/MyDrive/eye_project/my_dataset/dataset_B_Facial_Images.zip" -d /content/cew100
print("Ready!")

Mounted at /content/drive
Ready!


In [ ]:
# ===================== Train model v2 (blur augmentation) + test it on our test set and on CEW =====================
import csv, random, time
from math import atan2, degrees, hypot
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

PROJECT = "/content/drive/MyDrive/eye_project"
SPLIT_FILE = f"{PROJECT}/split.csv"
MODEL_V2 = f"{PROJECT}/models/eye_state_mnv3s_v2.pt"
FACE_MODEL = f"{PROJECT}/models/face_landmarker.task"
CEW = {
    0: "/content/cew100/dataset_B_Facial_Images/dataset_B_FacialImages/ClosedFace",
    1: "/content/cew100/dataset_B_Facial_Images/dataset_B_FacialImages/OpenFace",
}
CLASSES = ["closed", "open"]
IMG_SIZE = 112
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EPOCHS = 15
PRETRAINED = "IMAGENET1K_V1"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); random.seed(0)

# ---------- 1) transforms: the NEW part is RandomLowRes + blur ----------
class RandomLowRes:
    """Shrink the image to 20-56 px and enlarge it back -> blurry, low-resolution look (like CEW)."""
    def __init__(self, p=0.5, min_size=20, max_size=56):
        self.p, self.min_size, self.max_size = p, min_size, max_size
    def __call__(self, img):
        if random.random() < self.p:
            s = random.randint(self.min_size, self.max_size)
            w, h = img.size
            img = img.resize((s, s), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return img

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    RandomLowRes(p=0.5),                                                        # NEW
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.5, contrast=0.5),                      # was 0.4
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),   # NEW
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ---------- 2) data (same split as before) ----------
class EyeDataset(Dataset):
    def __init__(self, split, transform):
        with open(SPLIT_FILE) as f:
            self.items = [(r["path"], int(r["label"])) for r in csv.DictReader(f) if r["split"] == split]
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.transform(Image.open(path).convert("RGB")), label

train_ds, val_ds, test_ds = EyeDataset("train", train_transform), EyeDataset("val", eval_transform), EyeDataset("test", eval_transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=64, num_workers=2)
print(f"train {len(train_ds)}  val {len(val_ds)}  test {len(test_ds)}")

# ---------- 3) model: MobileNetV3-Small, last layer 1000 -> 2 ----------
model = models.mobilenet_v3_small(weights=PRETRAINED)
model.classifier[3] = nn.Linear(1024, 2)
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.momentum = 0.1
model = model.to(device)

# ---------- 4) training ----------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            seen += labels.size(0)
    return total_loss / seen, correct / seen

best_val = -1.0
Path(MODEL_V2).parent.mkdir(parents=True, exist_ok=True)
print("\n=== Training v2 ===")
for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(train_loader, True)
    vl, va = run_epoch(val_loader, False)
    scheduler.step()
    note = ""
    if va > best_val:
        best_val = va
        torch.save(model.state_dict(), MODEL_V2)
        note = "  <- best, saved"
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}{note}")

model.load_state_dict(torch.load(MODEL_V2, map_location=device))
model.eval()

def show_cm(y_true, y_pred, title, file_name):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap="Blues", values_format="d")
    plt.title(title)
    plt.savefig(f"{PROJECT}/{file_name}", dpi=150, bbox_inches="tight")
    plt.show()

# ---------- 5) our test set (971 eye images) ----------
preds = []
with torch.no_grad():
    for images, _ in test_loader:
        preds.append(model(images.to(device)).argmax(1).cpu())
preds = torch.cat(preds).numpy()
labels = np.array([l for _, l in test_ds.items])
test_acc = (preds == labels).mean()
print(f"\n=== OUR TEST SET: accuracy {test_acc:.4f} ({(preds == labels).sum()}/{len(labels)}) ===")
print(classification_report(labels, preds, labels=[0, 1], target_names=CLASSES, digits=4, zero_division=0))
show_cm(labels, preds, "Model v2 - our test set", "confusion_matrix_test_v2.png")

# ---------- 6) CEW (full faces -> MediaPipe -> eye crops -> model) ----------
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_buffer=open(FACE_MODEL, "rb").read()),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))

EYE_CORNERS = {"right": (33, 133), "left": (362, 263)}
CROP_SIZE, CROP_SCALE = 128, 2.2

def extract_eyes(frame_bgr, landmarks):
    h, w = frame_bgr.shape[:2]
    crops = []
    for i1, i2 in EYE_CORNERS.values():
        p1 = np.array([landmarks[i1].x * w, landmarks[i1].y * h])
        p2 = np.array([landmarks[i2].x * w, landmarks[i2].y * h])
        if p1[0] > p2[0]:
            p1, p2 = p2, p1
        center = (p1 + p2) / 2
        side = hypot(*(p2 - p1)) * CROP_SCALE
        angle = degrees(atan2(p2[1] - p1[1], p2[0] - p1[0]))
        M = cv2.getRotationMatrix2D((float(center[0]), float(center[1])), angle, CROP_SIZE / side)
        M[0, 2] += CROP_SIZE / 2 - center[0]
        M[1, 2] += CROP_SIZE / 2 - center[1]
        crops.append(cv2.warpAffine(frame_bgr, M, (CROP_SIZE, CROP_SIZE), borderMode=cv2.BORDER_REPLICATE))
    return crops

def p_open_of(crops):
    batch = torch.stack([eval_transform(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops]).to(device)
    with torch.no_grad():
        return float(torch.softmax(model(batch), 1)[:, 1].mean())

print("\n=== CEW: processing 2423 face images (takes a few minutes) ===")
cew_true, cew_pred, no_face = [], [], 0
for label_idx, folder in CEW.items():
    for path in sorted(Path(folder).glob("*")):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        pad = int(0.4 * max(frame.shape[:2]))
        frame = cv2.copyMakeBorder(frame, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(128, 128, 128))
        if max(frame.shape[:2]) < 400:
            s = 400 / max(frame.shape[:2])
            frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        if not res.face_landmarks:
            no_face += 1
            continue
        cew_true.append(label_idx)
        cew_pred.append(1 if p_open_of(extract_eyes(frame, res.face_landmarks[0])) > 0.5 else 0)

cew_true, cew_pred = np.array(cew_true), np.array(cew_pred)
cew_acc = (cew_true == cew_pred).mean()
cm = confusion_matrix(cew_true, cew_pred, labels=[0, 1])
print(f"no face found: {no_face}")
print(f"=== CEW: accuracy {cew_acc:.4f} ({(cew_true == cew_pred).sum()}/{len(cew_true)}) ===")
print(classification_report(cew_true, cew_pred, labels=[0, 1], target_names=CLASSES, digits=4, zero_division=0))
show_cm(cew_true, cew_pred, "Model v2 - CEW external test", "confusion_matrix_cew_v2.png")

# ---------- 7) before / after ----------
print("\n================ BEFORE vs AFTER ================")
print(f"{'':10s} {'Our test':>10s} {'CEW':>8s} {'CEW closed recall':>18s} {'CEW open recall':>16s}")
print(f"{'v1 (old)':10s} {0.9979:10.4f} {0.8785:8.4f} {1115/1189:18.4f} {1011/1231:16.4f}")
print(f"{'v2 (new)':10s} {test_acc:10.4f} {cew_acc:8.4f} {cm[0,0]/max(1,cm[0].sum()):18.4f} {cm[1,1]/max(1,cm[1].sum()):16.4f}")
print(f"\nModel v2 saved to: {MODEL_V2}  (the old model is unchanged)")

train 3747  val 734  test 971
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 92.8MB/s]


=== Training v2 ===


Epoch  1/15 | train loss 0.1247 acc 0.9490 | val loss 0.0105 acc 0.9986  <- best, saved
Epoch  2/15 | train loss 0.0091 acc 0.9968 | val loss 0.0021 acc 1.0000  <- best, saved
Epoch  3/15 | train loss 0.0077 acc 0.9976 | val loss 0.0035 acc 0.9986
Epoch  4/15 | train loss 0.0052 acc 0.9987 | val loss 0.0008 acc 1.0000
Epoch  5/15 | train loss 0.0070 acc 0.9976 | val loss 0.0044 acc 0.9986
Epoch  6/15 | train loss 0.0027 acc 0.9992 | val loss 0.0023 acc 0.9986
Epoch  7/15 | train loss 0.0009 acc 0.9997 | val loss 0.0017 acc 0.9986
Epoch  8/15 | train loss 0.0016 acc 0.9992 | val loss 0.0008 acc 1.0000


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="photo.jpg", delay_ms=3000):
    js = Javascript('''
    async function takePhoto(delayMs) {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Capture (photo after 3 seconds)';
      div.appendChild(button);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => button.onclick = resolve);
      button.textContent = 'Taking photo in 3 seconds...';
      await new Promise((resolve) => setTimeout(resolve, delayMs));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.9);
    }
    ''')
    display(js)
    data = eval_js(f"takePhoto({delay_ms})")
    with open(filename, "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    return filename

photo_path = take_photo()
print("Saved:", photo_path)

In [ ]:
!unzip -q -o "/content/drive/MyDrive/eye_project/my_dataset/dataset_B_FacialImages_highResolution.zip" -d /content/cew

HIGHRES = "/content/cew/dataset_B_FacialImages_highResolution/dataset_B_FacialImages_highResolution"
sizes, detected, closed_ok, no_face_hr = [], 0, 0, 0

for path in sorted(Path(HIGHRES).glob("*")):
    frame = cv2.imread(str(path))
    if frame is None:
        continue
    sizes.append(max(frame.shape[:2]))
    pad = int(0.4 * max(frame.shape[:2]))
    frame = cv2.copyMakeBorder(frame, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(128, 128, 128))
    if max(frame.shape[:2]) < 400:
        s = 400 / max(frame.shape[:2])
        frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    if not res.face_landmarks:
        no_face_hr += 1
        continue
    detected += 1
    if p_open_of(extract_eyes(frame, res.face_landmarks[0])) <= 0.5:
        closed_ok += 1

print(f"High-resolution closed faces: {len(sizes)} images, median size {np.median(sizes):.0f} px, no face: {no_face_hr}")
print(f"\nClosed eyes detected (same people, same photos):")
print(f"   100 x 100 version:       {cm[0, 0] / cm[0].sum():.4f}")
print(f"   original resolution:     {closed_ok / detected:.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score

# 6 points per eye: corner, top, top, corner, bottom, bottom
RIGHT_EAR = [33, 160, 158, 133, 153, 144]
LEFT_EAR = [362, 385, 387, 263, 373, 380]

def eye_aspect_ratio(lm, idx, w, h):
    p = [np.array([lm[i].x * w, lm[i].y * h]) for i in idx]
    vertical = np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])
    horizontal = np.linalg.norm(p[0] - p[3])
    return vertical / (2 * horizontal)

print("Processing CEW (takes a few minutes)...")
rows = []      # (true label, EAR, CNN P(open))
for label_idx, folder in CEW.items():
    for path in sorted(Path(folder).glob("*")):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        pad = int(0.4 * max(frame.shape[:2]))
        frame = cv2.copyMakeBorder(frame, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(128, 128, 128))
        if max(frame.shape[:2]) < 400:
            s = 400 / max(frame.shape[:2])
            frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        if not res.face_landmarks:
            continue
        lm = res.face_landmarks[0]
        h, w = frame.shape[:2]
        ear = (eye_aspect_ratio(lm, RIGHT_EAR, w, h) + eye_aspect_ratio(lm, LEFT_EAR, w, h)) / 2
        rows.append((label_idx, ear, p_open_of(extract_eyes(frame, lm))))

y = np.array([r[0] for r in rows])
ears = np.array([r[1] for r in rows])
probs = np.array([r[2] for r in rows])

def scores(pred):
    acc = (pred == y).mean()
    closed_recall = ((pred == 0) & (y == 0)).sum() / (y == 0).sum()
    open_recall = ((pred == 1) & (y == 1)).sum() / (y == 1).sum()
    return acc, closed_recall, open_recall

# EAR: standard threshold, and the BEST threshold found on CEW itself (optimistic for EAR)
thresholds = np.arange(0.05, 0.45, 0.005)
best_t = max(thresholds, key=lambda t: ((ears > t).astype(int) == y).mean())

results_table = {
    "EAR, threshold 0.20 (standard)": scores((ears > 0.20).astype(int)),
    f"EAR, best threshold {best_t:.3f}": scores((ears > best_t).astype(int)),
    "CNN v2 (ours), threshold 0.5": scores((probs > 0.5).astype(int)),
}

print(f"\nImages with a face: {len(y)}\n")
print(f"{'Method':34s} {'Accuracy':>9s} {'Closed recall':>14s} {'Open recall':>12s}")
for name, (a, c, o) in results_table.items():
    print(f"{name:34s} {a:9.4f} {c:14.4f} {o:12.4f}")
print(f"\nAUC (threshold-free):   EAR = {roc_auc_score(y, ears):.4f}    CNN v2 = {roc_auc_score(y, probs):.4f}")

# How well does EAR separate open from closed?
plt.figure(figsize=(8, 4))
plt.hist(ears[y == 0], bins=50, alpha=0.6, label="closed")
plt.hist(ears[y == 1], bins=50, alpha=0.6, label="open")
plt.axvline(0.20, color="k", linestyle="--", label="threshold 0.20")
plt.xlabel("EAR")
plt.ylabel("number of images")
plt.title("EAR distribution on CEW")
plt.legend()
plt.savefig("/content/drive/MyDrive/eye_project/ear_histogram_cew.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cv_probs_scaled(features):
    pipe = make_pipeline(StandardScaler(), LogisticRegression())
    return cross_val_predict(pipe, features, y, cv=cv, method="predict_proba")[:, 1]

candidates = {
    "EAR only (5-fold CV)":         cv_probs_scaled(ears.reshape(-1, 1)),
    "CNN v2 only (5-fold CV)":      cv_probs_scaled(probs.reshape(-1, 1)),
    "Hybrid EAR + CNN (5-fold CV)": cv_probs_scaled(np.column_stack([ears, probs])),
}

print(f"{'Method':30s} {'Accuracy':>9s} {'Closed recall':>14s} {'Open recall':>12s} {'AUC':>7s}")
for name, p in candidates.items():
    a, c, o = scores((p > 0.5).astype(int))
    print(f"{name:30s} {a:9.4f} {c:14.4f} {o:12.4f} {roc_auc_score(y, p):7.4f}")

final = make_pipeline(StandardScaler(), LogisticRegression()).fit(np.column_stack([ears, probs]), y)
scaler, lr = final.named_steps["standardscaler"], final.named_steps["logisticregression"]
print("\nScaler mean:", scaler.mean_, " scale:", scaler.scale_)
print("Weights: w_EAR = %.3f  w_CNN = %.3f  bias = %.3f" % (lr.coef_[0][0], lr.coef_[0][1], lr.intercept_[0]))

In [ ]:
# ===== Experiment: train the CNN ON CEW (70%) and test on the other 30% - the way papers compare =====
import random, json
from math import atan2, degrees, hypot
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

PROJECT = "/content/drive/MyDrive/eye_project"
FACE_MODEL = f"{PROJECT}/models/face_landmarker.task"
MODEL_V2 = f"{PROJECT}/models/eye_state_mnv3s_v2.pt"
MODEL_CEW = f"{PROJECT}/models/eye_state_mnv3s_cew.pt"
CEW = {
    0: "/content/cew100/dataset_B_Facial_Images/dataset_B_FacialImages/ClosedFace",
    1: "/content/cew100/dataset_B_Facial_Images/dataset_B_FacialImages/OpenFace",
}
IMG_SIZE, CROP_SIZE, CROP_SCALE = 112, 128, 2.2
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EPOCHS = 15
PRETRAINED = "IMAGENET1K_V1"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); random.seed(0); np.random.seed(0)

EYE_CORNERS = [(33, 133), (362, 263)]
RIGHT_EAR = [33, 160, 158, 133, 153, 144]
LEFT_EAR = [362, 385, 387, 263, 373, 380]

class RandomLowRes:
    def __init__(self, p=0.5, min_size=20, max_size=56):
        self.p, self.min_size, self.max_size = p, min_size, max_size
    def __call__(self, img):
        if random.random() < self.p:
            s = random.randint(self.min_size, self.max_size)
            w, h = img.size
            img = img.resize((s, s), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return img

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), RandomLowRes(p=0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.5, contrast=0.5),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

def ear_of(lm, idx, w, h):
    p = [np.array([lm[i].x * w, lm[i].y * h]) for i in idx]
    return (np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])) / (2 * np.linalg.norm(p[0] - p[3]))

def crops_of(frame, lm):
    h, w = frame.shape[:2]
    out = []
    for i1, i2 in EYE_CORNERS:
        p1 = np.array([lm[i1].x * w, lm[i1].y * h]); p2 = np.array([lm[i2].x * w, lm[i2].y * h])
        if p1[0] > p2[0]:
            p1, p2 = p2, p1
        c = (p1 + p2) / 2
        side = hypot(*(p2 - p1)) * CROP_SCALE
        M = cv2.getRotationMatrix2D((float(c[0]), float(c[1])), degrees(atan2(p2[1] - p1[1], p2[0] - p1[0])), CROP_SIZE / side)
        M[0, 2] += CROP_SIZE / 2 - c[0]; M[1, 2] += CROP_SIZE / 2 - c[1]
        out.append(cv2.warpAffine(frame, M, (CROP_SIZE, CROP_SIZE), borderMode=cv2.BORDER_REPLICATE))
    return out

# ---------- 1) run MediaPipe once on every CEW face: eye crops + EAR ----------
items = []      # (label, [right_crop, left_crop], EAR)
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision
landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_buffer=open(FACE_MODEL, "rb").read()),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))
print("Step 1/4: MediaPipe on all CEW faces (a few minutes)...")
for label, folder in CEW.items():
    for path in sorted(Path(folder).glob("*")):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        pad = int(0.4 * max(frame.shape[:2]))
        frame = cv2.copyMakeBorder(frame, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(128, 128, 128))
        if max(frame.shape[:2]) < 400:
            s = 400 / max(frame.shape[:2])
            frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        if not res.face_landmarks:
            continue
        lm = res.face_landmarks[0]
        h, w = frame.shape[:2]
        items.append((label, crops_of(frame, lm), (ear_of(lm, RIGHT_EAR, w, h) + ear_of(lm, LEFT_EAR, w, h)) / 2))
labels_all = np.array([it[0] for it in items])
ears_all = np.array([it[2] for it in items])
print(f"Faces found: {len(items)}")

# ---------- 2) split by face: 70% train (part of it for validation), 30% test ----------
idx = np.arange(len(items))
train_idx, test_idx = train_test_split(idx, test_size=0.30, stratify=labels_all, random_state=0)
fit_idx, val_idx = train_test_split(train_idx, test_size=0.15, stratify=labels_all[train_idx], random_state=0)
print(f"Step 2/4: train {len(fit_idx)} faces, validation {len(val_idx)} faces, TEST {len(test_idx)} faces")

class CropDataset(Dataset):
    def __init__(self, ids, transform):
        self.samples = [(crop, items[i][0]) for i in ids for crop in items[i][1]]     # 2 eyes per face
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, k):
        crop, label = self.samples[k]
        return self.transform(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))), label

fit_loader = DataLoader(CropDataset(fit_idx, train_transform), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(CropDataset(val_idx, eval_transform), batch_size=64, num_workers=2)

# ---------- 3) train MobileNetV3-Small on CEW eye crops (from ImageNet, like before) ----------
def new_model(weights):
    m = models.mobilenet_v3_small(weights=weights)
    m.classifier[3] = nn.Linear(1024, 2)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.momentum = 0.1
    return m.to(device)

model = new_model(PRETRAINED)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train):
    model.train(train)
    correct, seen, total_loss = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, yb in loader:
            x, yb = x.to(device), yb.to(device)
            out = model(x)
            loss = criterion(out, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * yb.size(0)
            correct += (out.argmax(1) == yb).sum().item(); seen += yb.size(0)
    return total_loss / seen, correct / seen

print("Step 3/4: training on CEW")
best = -1
for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(fit_loader, True)
    vl, va = run_epoch(val_loader, False)
    scheduler.step()
    note = ""
    if va > best:
        best = va; torch.save(model.state_dict(), MODEL_CEW); note = "  <- best, saved"
    print(f"Epoch {epoch:2d}/{EPOCHS} | train acc {ta:.4f} | val acc {va:.4f}{note}")

# ---------- 4) compare everything on the SAME 30% test faces ----------
def face_probs(m, ids):
    m.eval()
    crops = [crop for i in ids for crop in items[i][1]]
    x = torch.stack([eval_transform(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops])
    probs = []
    with torch.no_grad():
        for k in range(0, len(x), 128):
            probs.append(torch.softmax(m(x[k:k + 128].to(device)), 1)[:, 1].cpu())
    return torch.cat(probs).numpy().reshape(-1, 2).mean(1)        # average of both eyes

y_test = labels_all[test_idx]

def row(name, p_open, threshold, score_for_auc):
    pred = (p_open > threshold).astype(int)
    acc = (pred == y_test).mean()
    closed = ((pred == 0) & (y_test == 0)).sum() / (y_test == 0).sum()
    opened = ((pred == 1) & (y_test == 1)).sum() / (y_test == 1).sum()
    print(f"{name:42s} {acc:9.4f} {closed:14.4f} {opened:12.4f} {roc_auc_score(y_test, score_for_auc):7.4f}")
    return acc

model.load_state_dict(torch.load(MODEL_CEW, map_location=device))
p_cew = face_probs(model, test_idx)

model_v2 = new_model(None)
model_v2.load_state_dict(torch.load(MODEL_V2, map_location=device))
p_v2 = face_probs(model_v2, test_idx)

ear_test = ears_all[test_idx]
thresholds = np.arange(0.05, 0.45, 0.005)
t_train = max(thresholds, key=lambda t: ((ears_all[train_idx] > t).astype(int) == labels_all[train_idx]).mean())

print("\nStep 4/4: RESULTS on the same TEST faces (30% of CEW, never used for training)")
print(f"{'Method':42s} {'Accuracy':>9s} {'Closed recall':>14s} {'Open recall':>12s} {'AUC':>7s}")
row("CNN v2 (trained on OUR data, never saw CEW)", p_v2, 0.5, p_v2)
row("CNN trained on CEW train split", p_cew, 0.5, p_cew)
row("EAR, threshold 0.20 (standard)", ear_test, 0.20, ear_test)
row(f"EAR, threshold {t_train:.3f} (tuned on train split)", ear_test, t_train, ear_test)
print(f"\nCEW-trained model saved to: {MODEL_CEW}")

Step 1/4: MediaPipe on all CEW faces (a few minutes)...
Faces found: 2420
Step 2/4: train 1439 faces, validation 255 faces, TEST 726 faces
Step 3/4: training on CEW
Epoch  1/15 | train acc 0.8739 | val acc 0.9529  <- best, saved
Epoch  2/15 | train acc 0.9555 | val acc 0.9745  <- best, saved
Epoch  3/15 | train acc 0.9628 | val acc 0.9784  <- best, saved
Epoch  4/15 | train acc 0.9632 | val acc 0.9765
Epoch  5/15 | train acc 0.9705 | val acc 0.9706
Epoch  6/15 | train acc 0.9757 | val acc 0.9843  <- best, saved
Epoch  7/15 | train acc 0.9698 | val acc 0.9686
Epoch  8/15 | train acc 0.9753 | val acc 0.9804
Epoch  9/15 | train acc 0.9736 | val acc 0.9686
Epoch 10/15 | train acc 0.9778 | val acc 0.9765
Epoch 11/15 | train acc 0.9847 | val acc 0.9745
Epoch 12/15 | train acc 0.9840 | val acc 0.9784
Epoch 13/15 | train acc 0.9889 | val acc 0.9765
Epoch 14/15 | train acc 0.9858 | val acc 0.9765
Epoch 15/15 | train acc 0.9851 | val acc 0.9765

Step 4/4: RESULTS on the same TEST faces (30% of C

In [ ]:
# ===== Model v3: train on OUR data + CEW train split, then test on BOTH test sets =====
# Run this right after the CEW experiment cell (it reuses its variables).
import csv
from torch.utils.data import ConcatDataset

SPLIT_FILE = f"{PROJECT}/split.csv"
MODEL_V3 = f"{PROJECT}/models/eye_state_mnv3s_v3.pt"

class OurDataset(Dataset):
    def __init__(self, split, transform):
        with open(SPLIT_FILE) as f:
            self.items = [(r["path"], int(r["label"])) for r in csv.DictReader(f) if r["split"] == split]
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.transform(Image.open(path).convert("RGB")), label

# ---------- 1) data: our train/val + CEW train/val (the CEW test faces stay untouched) ----------
train_set = ConcatDataset([OurDataset("train", train_transform), CropDataset(fit_idx, train_transform)])
val_set = ConcatDataset([OurDataset("val", eval_transform), CropDataset(val_idx, eval_transform)])
fit_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=64, num_workers=2)
print(f"Step 1/3: train {len(train_set)} eye images (ours + CEW), validation {len(val_set)}")

# ---------- 2) train v3 (same model and settings as before) ----------
torch.manual_seed(0); random.seed(0)
model = new_model(PRETRAINED)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Step 2/3: training v3")
best = -1
for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(fit_loader, True)
    vl, va = run_epoch(val_loader, False)
    scheduler.step()
    note = ""
    if va > best:
        best = va; torch.save(model.state_dict(), MODEL_V3); note = "  <- best, saved"
    print(f"Epoch {epoch:2d}/{EPOCHS} | train acc {ta:.4f} | val acc {va:.4f}{note}")
model.load_state_dict(torch.load(MODEL_V3, map_location=device))
model.eval()

# ---------- 3) test on BOTH test sets ----------
our_test = OurDataset("test", eval_transform)
y_our = np.array([l for _, l in our_test.items])

def our_preds(m):
    m.eval()
    preds = []
    with torch.no_grad():
        for x, _ in DataLoader(our_test, batch_size=128, num_workers=2):
            preds.append(m(x.to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy()

def summary(name, pred, y):
    acc = (pred == y).mean()
    closed = ((pred == 0) & (y == 0)).sum() / (y == 0).sum()
    opened = ((pred == 1) & (y == 1)).sum() / (y == 1).sum()
    print(f"{name:34s} {acc:9.4f} {closed:14.4f} {opened:12.4f}")

model_cew = new_model(None)
model_cew.load_state_dict(torch.load(MODEL_CEW, map_location=device))

header = f"{'Method':34s} {'Accuracy':>9s} {'Closed recall':>14s} {'Open recall':>12s}"
print("\nStep 3/3: RESULTS")
print(f"\n--- OUR TEST SET ({len(y_our)} eye images, unseen persons) ---")
print(header)
summary("CNN v2 (our data only)", our_preds(model_v2), y_our)
summary("CNN trained on CEW only", our_preds(model_cew), y_our)
summary("CNN v3 (our data + CEW)", our_preds(model), y_our)

p_v3 = face_probs(model, test_idx)
print(f"\n--- CEW TEST ({len(y_test)} faces, never used for training) ---")
print(header)
summary("CNN v2 (our data only)", (p_v2 > 0.5).astype(int), y_test)
summary("CNN trained on CEW only", (p_cew > 0.5).astype(int), y_test)
summary("CNN v3 (our data + CEW)", (p_v3 > 0.5).astype(int), y_test)
summary(f"EAR, threshold {t_train:.3f}", (ear_test > t_train).astype(int), y_test)
print(f"\nAUC on CEW test:  v3 = {roc_auc_score(y_test, p_v3):.4f}   EAR = {roc_auc_score(y_test, ear_test):.4f}")
print(f"\nModel v3 saved to: {MODEL_V3}")

Step 1/3: train 6625 eye images (ours + CEW), validation 1244
Step 2/3: training v3
Epoch  1/15 | train acc 0.9390 | val acc 0.9847  <- best, saved
Epoch  2/15 | train acc 0.9795 | val acc 0.9871  <- best, saved
Epoch  3/15 | train acc 0.9846 | val acc 0.9920  <- best, saved
Epoch  4/15 | train acc 0.9857 | val acc 0.9904
Epoch  5/15 | train acc 0.9870 | val acc 0.9887
Epoch  6/15 | train acc 0.9861 | val acc 0.9904
Epoch  7/15 | train acc 0.9891 | val acc 0.9912
Epoch  8/15 | train acc 0.9903 | val acc 0.9871
Epoch  9/15 | train acc 0.9929 | val acc 0.9895
Epoch 10/15 | train acc 0.9920 | val acc 0.9920
Epoch 11/15 | train acc 0.9943 | val acc 0.9912
Epoch 12/15 | train acc 0.9934 | val acc 0.9904
Epoch 13/15 | train acc 0.9941 | val acc 0.9904
Epoch 14/15 | train acc 0.9946 | val acc 0.9912
Epoch 15/15 | train acc 0.9947 | val acc 0.9904

Step 3/3: RESULTS

--- OUR TEST SET (971 eye images, unseen persons) ---
Method                              Accuracy  Closed recall  Open recall
C